<a href="https://colab.research.google.com/github/OutisAyo/council-classifier-/blob/main/notebooks/18_hybrid_urgency_layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning for Automated Classification and Prioritisation of Local Council Service Requests in the UK
## Notebook 18  Hybrid Urgency Layer

**Author:** Fashina Fuad Ayomide  
**MSc Data Science, University of South Wales**

Diagnostic analysis showed that explicit urgency language appears in only 0.12% of records, too sparse to train a reliable urgency classifier. This notebook instead adds a transparent, rule-based urgency override layer on top of the resolution-time priority model, combining machine learning with auditable rules — a common pattern in real-world triage systems.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import joblib

models_dir = '/content/drive/MyDrive/council-classifier/models'

department_pipeline = joblib.load(f'{models_dir}/department_pipeline_rf.pkl')
priority_pipeline = joblib.load(f'{models_dir}/priority_pipeline_final.pkl')

print("Pipelines loaded.")

Pipelines loaded.


## Building the urgency lexicon

We define urgency signals in three categories, each reflecting a different kind of urgency a citizen might express.

In [11]:
URGENCY_LEXICON = {
    # Explicit urgency statements
    'explicit': [
        'urgent', 'emergency', 'immediately', 'immediate', 'asap',
        'right now', 'straight away', 'critical', 'cant wait', "can't wait"
    ],

    # Physical hazards and dangerous conditions
    # (blocked drain, sewer, hazard, warning, chemical validated from the data)
    'hazard': [
        'dangerous', 'danger', 'hazard', 'hazardous', 'warning',
        'fire', 'smoke', 'flood', 'flooding', 'leak', 'leaking',
        'gas', 'chemical', 'chem', 'electrical', 'sparking',
        'collapse', 'collapsed', 'fallen', 'falling', 'unsafe',
        'blocked drain', 'sewer', 'sewage', 'overflow', 'overflowing',
        'exposed', 'structural', 'subsidence', 'asbestos',
        'needle', 'syringe', 'sharp', 'broken glass'
    ],

    # Human distress, vulnerability, and health impact
    'distress': [
        'cant sleep', "can't sleep", 'cannot sleep', 'please help',
        'please solve', 'desperate', 'suffering', 'unbearable',
        'cannot cope', 'cant cope', 'at risk', 'scared', 'frightened',
        'no water', 'no heating', 'breathing',
        'child', 'baby', 'elderly', 'disabled', 'vulnerable',
        'pregnant', 'ill', 'sick', 'injured', 'injury'
    ]
}

# Flatten into one list for simple checking
ALL_URGENCY_WORDS = [word for group in URGENCY_LEXICON.values() for word in group]

print("Total urgency terms:", len(ALL_URGENCY_WORDS))
print("\nBy category:")
for category, words in URGENCY_LEXICON.items():
    print(f"  {category}: {len(words)} terms")

Total urgency terms: 70

By category:
  explicit: 10 terms
  hazard: 34 terms
  distress: 26 terms


## The urgency detection function

Given a request text, this checks whether any urgency term is present and returns which ones matched.

In [12]:
def detect_urgency(text):
    text_lower = text.lower()
    matched = [word for word in ALL_URGENCY_WORDS if word in text_lower]
    return matched

# Quick test
print(detect_urgency("animals barking all night, cant sleep at all please solve this"))
print(detect_urgency("recycling bag not collected"))

['cant sleep', 'please solve']
[]


## The hybrid priority function

This combines the ML model's prediction with the urgency rule layer. If urgency language is detected, the priority is escalated.

In [13]:
def get_hybrid_priority(text, day_of_week, month):
    # Step 1: ML model's resolution-time prediction
    model_input = pd.DataFrame({
        'request_text': [text],
        'day_of_week': [day_of_week],
        'month': [month]
    })
    model_priority = priority_pipeline.predict(model_input)[0]

    # Step 2: rule-based urgency check
    urgency_matches = detect_urgency(text)

    # Step 3: escalate if urgency found
    if urgency_matches:
        final_priority = 'HIGH'
        reason = f"Escalated to HIGH — urgency language detected: {urgency_matches}"
    else:
        final_priority = model_priority
        reason = "Based on resolution-time model prediction"

    return {
        'model_priority': model_priority,
        'final_priority': final_priority,
        'urgency_matches': urgency_matches,
        'reason': reason
    }

## Testing against the known problem cases

We test the two examples the original model got wrong, plus a routine case that should stay unaffected.

In [14]:
test_cases = [
    ("animals barking all night, cant sleep at all please solve this", 1, 7),
    ("there is a dangerous gas leak in my street", 2, 7),
    ("recycling bag not collected", 4, 3),
    ("can i have any information regarding how to license my vet clinic", 0, 7),
]

for text, dow, month in test_cases:
    result = get_hybrid_priority(text, dow, month)
    print(f"Request: '{text}'")
    print(f"  Model said: {result['model_priority']}  →  Final: {result['final_priority']}")
    print(f"  Reason: {result['reason']}")
    print()

Request: 'animals barking all night, cant sleep at all please solve this'
  Model said: LOW  →  Final: HIGH
  Reason: Escalated to HIGH — urgency language detected: ['cant sleep', 'please solve']

Request: 'there is a dangerous gas leak in my street'
  Model said: HIGH  →  Final: HIGH
  Reason: Escalated to HIGH — urgency language detected: ['dangerous', 'danger', 'leak', 'gas']

Request: 'recycling bag not collected'
  Model said: LOW  →  Final: LOW
  Reason: Based on resolution-time model prediction

Request: 'can i have any information regarding how to license my vet clinic'
  Model said: HIGH  →  Final: HIGH
  Reason: Based on resolution-time model prediction



## Saving the urgency lexicon

We save the lexicon so the dashboard can load and use the same urgency detection logic.

In [15]:
import pickle

with open(f'{models_dir}/urgency_lexicon.pkl', 'wb') as f:
    pickle.dump(URGENCY_LEXICON, f)

print("Urgency lexicon saved.")

Urgency lexicon saved.


In [16]:
import pandas as pd
from collections import Counter
import re

processed_dir = '/content/drive/MyDrive/council-classifier/processed'
dataset = pd.read_csv(f'{processed_dir}/swansea_cleaned.csv')

# --- Step 1: identify the urgency-flagged records ---
urgency_pattern = r'\*\s*urgent\s*\*|urgent|emergency|dangerous|danger|hazard'
flagged = dataset[dataset['request_text'].str.contains(urgency_pattern, case=False, na=False, regex=True)]

print("Urgency-flagged records:", len(flagged))

# --- Step 2: what words actually appear in these flagged requests? ---
def get_words(series):
    text = ' '.join(series.dropna().str.lower())
    words = re.findall(r'\b[a-z]+\b', text)   # letters-only words
    return Counter(words)

flagged_words = get_words(flagged['request_text'])
all_words = get_words(dataset['request_text'])

# --- Step 3: find words that are DISTINCTIVE to urgent requests ---
# A word is a good urgency signal if it appears often in flagged records
# AND is proportionally much more common there than in the dataset overall.
print("\nWords most distinctive to urgency-flagged requests:\n")
print(f"{'word':<20}{'in_flagged':<12}{'in_all':<10}{'lift':<8}")
print("-" * 50)

candidates = []
for word, flagged_count in flagged_words.most_common(100):
    if len(word) < 3:
        continue
    all_count = all_words[word]
    # proportion in flagged vs proportion in whole dataset
    flagged_prop = flagged_count / len(flagged)
    all_prop = all_count / len(dataset)
    lift = flagged_prop / all_prop if all_prop > 0 else 0
    if lift > 3 and flagged_count >= 5:   # much more common in urgent + not too rare
        candidates.append((word, flagged_count, all_count, lift))

for word, fc, ac, lift in sorted(candidates, key=lambda x: x[3], reverse=True):
    print(f"{word:<20}{fc:<12}{ac:<10}{lift:<8.1f}")

Urgency-flagged records: 337

Words most distinctive to urgency-flagged requests:

word                in_flagged  in_all    lift    
--------------------------------------------------
urgent              194         194       754.8   
live                170         170       754.8   
blocked             24          24        754.8   
drain               24          24        754.8   
sewer               24          24        754.8   
hazard              24          24        754.8   
warning             10          10        754.8   
chem                6           6         754.8   
dangerous           119         119       754.8   
trees               98          101       732.4   
dogs                15          30        377.4   
house               175         758       174.3   
rat                 170         867       148.0   
warden              95          2185      32.8    
animal              100         3930      19.2    
hours               44          3534      9.4     